# Fire-Weather Integration: ERA5 + LANDFIRE

**Project:** `landfire-exploration`  
**Study area:** Northern Colorado Mountain Landscape  
**LANDFIRE version:** LF2024 (version 240)  
**Fire-weather date:** August 13, 2020 (active fire-weather across the Rocky Mountain region)

---

## Conceptual framework

This notebook adds dynamic atmospheric conditions to the static LANDFIRE landscape.

```
LANDFIRE (relatively static, landscape state)
  Fuel Vegetation Type/Cover/Height  →  vegetation structure
  + disturbance history
  ↓
  FBFM40 fuel representation
  + topography (elevation, slope, aspect)

ERA5 (dynamic, updated hourly/daily)
  temperature, humidity, wind, VPD
  ↓
  atmospheric conditions / fuel moisture environment

Together: the inputs relevant to fire-behavior modeling
  (not a fire-risk score — a description of the physical environment)
```

**Key message:** A Scott & Burgan fuel model describes *how* surface fuels would burn under standardized fire-behavior modeling assumptions. Whether and how fast fire spreads also depends on fuel moisture (driven by weather and aspect) and wind. These are separate dynamic inputs — not encoded in the static fuel model.

---

**Pre-requisites:**  
Run `01_landfire_exploration.ipynb` first (provides processed rasters and DataFrame).

## 1  Setup

In [ ]:
import sys
from pathlib import Path

_cwd = Path.cwd()
PROJECT_ROOT = _cwd if (_cwd / 'src').exists() else _cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import xarray as xr
import rioxarray  # noqa

from src import (
    PROJECT_ROOT, RAW_DIR, PROCESSED_DIR, FIGURES_DIR, RESULTS_DIR, DATA_DIR,
    BBOX_WGS84, STUDY_AREA_NAME, ERA5_DATE, ERA5_HOUR, ERA5_VARIABLES,
)
from src.load_data import FBFM40_GROUP_COLORS, fbfm40_group
from src.analysis import raster_to_dataframe, plot_landscape_maps

%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})

print(f'Fire-weather date : {ERA5_DATE} {ERA5_HOUR:02d}:00 UTC')
print(f'ERA5 variables    : {ERA5_VARIABLES}')

## 2  Load Processed LANDFIRE Data

Load the aligned layers saved in notebook 01.

In [ ]:
from src.load_data import find_layer_file
from src.align_rasters import build_aligned_dataset

layers = {}
for var in ('evt', 'evc', 'evh', 'fuel_model', 'disturbance', 'elevation', 'slope', 'aspect'):
    p = PROCESSED_DIR / f'{var}.tif'
    if p.exists():
        da = xr.open_dataarray(p, engine='rasterio').squeeze('band', drop=True)
        da.name = var
        layers[var] = da
        print(f'  Loaded: {var}  shape={da.shape}')
    else:
        print(f'  [MISSING] {var} — run notebook 01 first')

if not layers:
    raise FileNotFoundError('No processed layers found. Run notebook 01 first.')

ds = xr.Dataset(layers)
print(f'\nDataset: {list(ds.data_vars)}')

In [ ]:
df = raster_to_dataframe(ds, max_pixels=300_000)
df['fuel_group'] = df['fuel_model'].apply(
    lambda c: fbfm40_group(int(c)) if not np.isnan(c) else 'Unknown'
)
print(f'Cell DataFrame: {len(df):,} rows')

## 3  ERA5 Fire-Weather Data

ERA5 is a global atmospheric reanalysis produced by ECMWF at 0.25° × 0.25° resolution (~27.8 km).
For this small study area (~36 × 39 km), ERA5 provides 2–4 grid cells — very coarse relative to
the 30 m LANDFIRE grid. This is intentional: it illustrates the fundamental scale mismatch between
atmospheric forcing data and landscape-scale fuel maps.

### ERA5 access options

**Option A — ARCO ERA5 (preferred; no credentials required):**  
Google hosts a publicly accessible ARCO ERA5 zarr on GCS. This is the same method used in
`climate-ml-data-pipeline` and requires no account or API key.

**Option B — CDS API** (requires free account at https://cds.climate.copernicus.eu):  
Set `USE_CDS = True` and configure `~/.cdsapirc` with your UID and API key.

**Option C — Pre-downloaded file:**  
Place an ERA5 netCDF at `data/raw/era5_fire_weather.nc` and both options above will be skipped.

In [ ]:
# ARCO ERA5 — publicly accessible, no credentials needed
# Same zarr store used in climate-ml-data-pipeline
ARCO_ERA5_PATH = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"

# Fire-weather variables (single-level names in ARCO ERA5)
ARCO_VARIABLES = [
    "2m_temperature",
    "2m_dewpoint_temperature",
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
]

# CDS fallback — set True only if ARCO is unavailable
USE_CDS = False

ERA5_OUTPUT = DATA_DIR / 'raw' / 'era5_fire_weather.nc'

In [ ]:
import os
os.environ.setdefault("GRPC_VERBOSITY", "ERROR")   # suppress gRPC fork noise in Jupyter
os.environ.setdefault("GRPC_TRACE", "")


def load_era5_arco(date: str, hour: int, bbox_wgs84: tuple, variables: list) -> xr.Dataset:
    """Load ERA5 fire-weather variables from the public ARCO GCS zarr.

    No credentials required — uses anonymous GCS access.
    Longitude convention in ARCO ERA5 is 0–360, so Western longitudes
    are converted before subsetting.
    """
    lon_min, lat_min, lon_max, lat_max = bbox_wgs84

    # ARCO uses 0–360 longitude; convert from signed degrees
    lon_min_360 = lon_min % 360
    lon_max_360 = lon_max % 360

    target_time = pd.Timestamp(f"{date}T{hour:02d}:00")

    print(f"Opening ARCO ERA5 zarr (anonymous GCS access)...")
    ds = xr.open_zarr(
        ARCO_ERA5_PATH,
        consolidated=True,
        storage_options={"token": "anon"},
        chunks=None,  # load eagerly — no dask needed for a small spatial subset
    )

    available = [v for v in variables if v in ds.data_vars]
    missing   = [v for v in variables if v not in ds.data_vars]
    if missing:
        print(f"  Variables not in this zarr: {missing}")
    if not available:
        raise KeyError(f"None of {variables} found in ARCO ERA5.")

    # ERA5 latitude is stored descending (90 → -90), so slice high → low
    subset = ds[available].sel(
        time=target_time,
        latitude=slice(lat_max + 0.5, lat_min - 0.5),
        longitude=slice(lon_min_360 - 0.5, lon_max_360 + 0.5),
    )
    print(f"  Loaded: {list(subset.data_vars)}  shape: {dict(subset.sizes)}")
    print(f"  Lat: {float(subset.latitude.min()):.2f}–{float(subset.latitude.max()):.2f}  "
          f"Lon: {float(subset.longitude.min()):.2f}–{float(subset.longitude.max()):.2f} (0–360)")
    return subset


def download_era5_cds(date: str, hour: int, bbox_wgs84: tuple, variables: list,
                      output_path: Path) -> bool:
    """Fallback: download ERA5 via CDS API. Requires ~/.cdsapirc."""
    try:
        import cdsapi
    except ImportError:
        print("cdsapi not installed. pip install cdsapi")
        return False

    lon_min, lat_min, lon_max, lat_max = bbox_wgs84
    area = [lat_max + 1.0, lon_min - 1.0, lat_min - 1.0, lon_max + 1.0]
    year, month, day = date.split("-")

    print(f"Downloading ERA5 via CDS API: {date} {hour:02d}:00 UTC")
    c = cdsapi.Client(quiet=True)
    c.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": "reanalysis",
            "variable":     variables,
            "year":  year, "month": month, "day": day,
            "time":  f"{hour:02d}:00",
            "area":  area,
            "format": "netcdf",
        },
        str(output_path),
    )
    return True

In [ ]:
era5_loaded = False
ds_era5 = None

# Option C: cached file
if ERA5_OUTPUT.exists():
    print(f"Loading cached ERA5 from {ERA5_OUTPUT}")
    ds_era5 = xr.open_dataset(ERA5_OUTPUT)
    era5_loaded = True

# Option A: ARCO public zarr (no credentials)
if not era5_loaded:
    try:
        ds_era5 = load_era5_arco(ERA5_DATE, ERA5_HOUR, BBOX_WGS84, ARCO_VARIABLES)
        ERA5_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
        ds_era5.to_netcdf(ERA5_OUTPUT)
        print(f"Cached to {ERA5_OUTPUT}")
        era5_loaded = True
    except Exception:
        import traceback
        traceback.print_exc()

# Option B: CDS API fallback
if not era5_loaded and USE_CDS:
    ERA5_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    success = download_era5_cds(
        ERA5_DATE, ERA5_HOUR, BBOX_WGS84,
        [v.replace("_", " ") for v in ARCO_VARIABLES],
        ERA5_OUTPUT,
    )
    if success:
        ds_era5 = xr.open_dataset(ERA5_OUTPUT)
        era5_loaded = True

if not era5_loaded:
    print("\nERA5 unavailable — proceeding with synthetic demo values.")
    ds_era5 = None
else:
    print(f"\nERA5 dataset:\n{ds_era5}")

## 4  Derive Fire-Weather Variables

From the raw ERA5 fields, we derive:
- **Temperature (°C)** — from 2m temperature (K)
- **Relative humidity (%)** — from temperature and dewpoint using the Magnus formula
- **Wind speed (m/s)** — from u and v 10m wind components
- **VPD (hPa)** — vapour pressure deficit; high VPD = dry, fire-prone conditions

In [ ]:
def kelvin_to_celsius(T_K):
    return T_K - 273.15

def magnus_saturation_vp(T_C):
    """Saturation vapour pressure (hPa) via Magnus formula."""
    return 6.1078 * np.exp(17.27 * T_C / (T_C + 237.3))

def relative_humidity(T_K, Td_K):
    """Relative humidity (%) from temperature and dewpoint in Kelvin."""
    T_C  = kelvin_to_celsius(T_K)
    Td_C = kelvin_to_celsius(Td_K)
    return 100 * magnus_saturation_vp(Td_C) / magnus_saturation_vp(T_C)

def vpd(T_K, Td_K):
    """Vapour pressure deficit (hPa)."""
    T_C  = kelvin_to_celsius(T_K)
    Td_C = kelvin_to_celsius(Td_K)
    es   = magnus_saturation_vp(T_C)
    e    = magnus_saturation_vp(Td_C)
    return es - e


if ds_era5 is not None:
    # ARCO ERA5 variable names; squeeze to 2D if time dim present
    era5_t = ds_era5.squeeze() if "time" in ds_era5.dims else ds_era5

    # Map ARCO names → arrays (fall back to CDS short names if loading from cached CDS file)
    def _get(ds, *candidates):
        for k in candidates:
            if k in ds:
                return ds[k].values
        raise KeyError(f"None of {candidates} found in ERA5 dataset")

    T_K  = _get(era5_t, "2m_temperature",         "t2m", "VAR_2T")
    Td_K = _get(era5_t, "2m_dewpoint_temperature", "d2m", "VAR_2D")
    u10  = _get(era5_t, "10m_u_component_of_wind", "u10", "VAR_10U")
    v10  = _get(era5_t, "10m_v_component_of_wind", "v10", "VAR_10V")

    temp_C   = kelvin_to_celsius(T_K)
    rh_pct   = relative_humidity(T_K, Td_K)
    wind_spd = np.sqrt(u10**2 + v10**2)
    vpd_hPa  = vpd(T_K, Td_K)

    lat_key = "latitude" if "latitude" in era5_t.coords else "lat"
    lon_key = "longitude" if "longitude" in era5_t.coords else "lon"
    era5_lat = era5_t[lat_key].values
    era5_lon = era5_t[lon_key].values % 360 - 360  # convert 0–360 → signed for display

    print(f"Temperature  : {temp_C.mean():.1f} °C (range: {temp_C.min():.1f}–{temp_C.max():.1f})")
    print(f"Rel. humidity: {rh_pct.mean():.0f}% (range: {rh_pct.min():.0f}–{rh_pct.max():.0f})")
    print(f"Wind speed   : {wind_spd.mean():.1f} m/s (max: {wind_spd.max():.1f})")
    print(f"VPD          : {vpd_hPa.mean():.1f} hPa (max: {vpd_hPa.max():.1f})")
else:
    print("ERA5 not loaded — using synthetic demo values.")
    temp_C   = np.array([[32., 30.], [33., 31.]])
    rh_pct   = np.array([[18., 20.], [17., 19.]])
    wind_spd = np.array([[ 8.,  6.], [ 9.,  7.]])
    vpd_hPa  = np.array([[22., 19.], [24., 20.]])
    era5_lat = np.array([40.5, 40.25])
    era5_lon = np.array([-105.75, -105.5])

## 5  Spatial Scale Comparison: ERA5 vs. LANDFIRE

ERA5 at 0.25° covers ~27.8 km per grid cell. The study area fits within 2–4 ERA5 cells.
This comparison illustrates a fundamental constraint in downscaling atmospheric data to
landscape-scale fuel maps: the meteorological forcing is nearly spatially uniform at the
scale where fuels vary over tens of metres.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: FBFM40 fuel groups as spatial context
ax = axes[0]
if 'fuel_model' in ds.data_vars:
    from src.analysis import _fbfm40_group_colormap
    arr = ds['fuel_model'].values.astype(float)
    arr[arr <= -9000] = np.nan
    rgba, legend = _fbfm40_group_colormap(arr)
    ax.imshow(rgba, origin='upper', interpolation='nearest')
    patches = [mpatches.Patch(facecolor=c, label=lbl) for lbl, c in legend.items()
               if lbl != 'Non-burnable']
    ax.legend(handles=patches, loc='lower right', fontsize=7, framealpha=0.85)
ax.set_title('LANDFIRE FBFM40 Fuel Groups (30 m)', fontsize=11, fontweight='bold')
ax.set_xlabel('West ← → East   (each pixel = 30 m)')
ax.axis('off')

# Right: ERA5 grid overlaid (schematic)
ax2 = axes[1]
# Show temperature as background
if temp_C.size > 1:
    extent_approx = [0, 10, 0, 10]  # normalized plot space
    im = ax2.imshow(temp_C, cmap='RdYlBu_r', origin='upper',
                    interpolation='nearest', aspect='auto')
    plt.colorbar(im, ax=ax2, shrink=0.8, label='Temperature (°C)')
    # Annotate with wind speed and RH
    for i in range(temp_C.shape[0]):
        for j in range(temp_C.shape[1]):
            ax2.text(
                j, i,
                f'{temp_C[i,j]:.0f}°C\nRH {rh_pct[i,j]:.0f}%\n{wind_spd[i,j]:.0f} m/s',
                ha='center', va='center', fontsize=9, fontweight='bold', color='black'
            )
ax2.set_title(
    f'ERA5 Fire-Weather Grid (~27.8 km cells)\n{ERA5_DATE} {ERA5_HOUR:02d}:00 UTC',
    fontsize=11, fontweight='bold'
)
ax2.set_xlabel('Each cell ≈ 27.8 km (covers most or all of the study area)')
ax2.set_xticks([]); ax2.set_yticks([])

fig.suptitle(
    'Spatial Scale: LANDFIRE (30 m, static) vs. ERA5 (27.8 km, hourly)',
    fontsize=12, fontweight='bold',
)
fig.tight_layout()
out = FIGURES_DIR / 'scale_comparison.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

![Spatial scale comparison: LANDFIRE (30m) vs ERA5 (27.8km)](../figures/scale_comparison.png)

## 6  Fuel × Weather Overlap Analysis

Because ERA5 provides only 4–5 cells across this study area, atmospheric variation is minimal — ERA5 essentially gives a single set of conditions for the whole landscape. The ERA5 mean is assigned uniformly to all LANDFIRE cells, which is appropriate at this scale and consistent with the scale-mismatch point illustrated in section 5.

What this section shows: how the fire-weather snapshot (temperature, RH, wind, VPD) sits relative to each fuel group — not spatial variation within the study area, but the question of *which fuels are present* under these conditions on this day.

In [ ]:
# Assign ERA5 fire-weather values to the LANDFIRE DataFrame.
#
# ERA5 at 0.25° (~27.8 km) gives 4–5 grid cells across this entire study area.
# Spatial variation within those cells is negligible relative to the 30 m LANDFIRE grid.
# Using the spatial mean is appropriate — and consistent with the scale-mismatch point
# made in section 5. Bilinear interpolation would be false precision here.

df['temp_C']   = float(np.nanmean(temp_C))
df['rh_pct']   = float(np.nanmean(rh_pct))
df['wind_spd'] = float(np.nanmean(wind_spd))
df['vpd_hPa']  = float(np.nanmean(vpd_hPa))

print(f"ERA5 mean conditions assigned to all LANDFIRE cells ({ERA5_DATE} {ERA5_HOUR:02d}:00 UTC):")
print(f"  Temperature : {df['temp_C'].iloc[0]:.1f} °C")
print(f"  Rel. humidity: {df['rh_pct'].iloc[0]:.0f} %")
print(f"  Wind speed  : {df['wind_spd'].iloc[0]:.1f} m/s")
print(f"  VPD         : {df['vpd_hPa'].iloc[0]:.1f} hPa")
print()
print(df[['fuel_group', 'temp_C', 'rh_pct', 'wind_spd', 'vpd_hPa']].head())

In [ ]:
# Summary: fire-weather conditions by fuel group
weather_by_group = (
    df.dropna(subset=['fuel_group', 'temp_C'])
    .groupby('fuel_group')[['temp_C', 'rh_pct', 'wind_spd', 'vpd_hPa']]
    .mean()
    .round(1)
)
print('Mean fire-weather conditions by fuel model group (study area, ', ERA5_DATE, '):')
print(weather_by_group.to_string())

## 7  Integrated Fire-Behavior Environment Visualization

This figure expresses the full conceptual hierarchy from the project framework:

```
Vegetation (EVT + EVC + EVH) + disturbance → FBFM40
FBFM40 + terrain (slope, aspect) + weather (Temp, RH, Wind, VPD)
→ inputs relevant to fire-behavior modeling
```

In [ ]:
import matplotlib

fig = plt.figure(figsize=(18, 14))
gs = fig.add_gridspec(3, 4, hspace=0.40, wspace=0.35)

# ── Row 1: Vegetation inputs ────────────────────────────────────────────────
for col_idx, (var, label, cmap) in enumerate([
    ('evt',  'Fuel Vegetation Type', 'tab20c'),
    ('evc',  'Fuel Vegetation Cover (%)', 'YlGn'),
    ('evh',  'Fuel Vegetation Height (m)', 'BuGn'),
    ('disturbance', 'Fuel Disturbance', 'RdGy_r'),
]):
    ax = fig.add_subplot(gs[0, col_idx])
    if var in ds.data_vars:
        arr = ds[var].values.astype(float)
        arr[arr <= -9000] = np.nan
        if var == 'disturbance':
            arr_plot = np.where(np.isnan(arr), np.nan, (arr > 0).astype(float))
            ax.imshow(arr_plot, cmap='RdYlGn', origin='upper', interpolation='nearest',
                      vmin=0, vmax=1)
        elif var == 'evt':
            uniq = np.unique(arr[~np.isnan(arr)])
            cmap_base = matplotlib.colormaps.get_cmap('tab20c').resampled(len(uniq))
            code_to_idx = {int(c): i for i, c in enumerate(uniq)}
            mapped = np.full_like(arr, np.nan)
            for c, i in code_to_idx.items():
                mapped[arr == c] = i
            ax.imshow(mapped, cmap=cmap_base, origin='upper', interpolation='nearest')
        else:
            vmin = np.nanpercentile(arr, 2)
            vmax = np.nanpercentile(arr, 98)
            ax.imshow(arr, cmap=cmap, origin='upper', interpolation='nearest',
                      vmin=vmin, vmax=vmax)
    ax.set_title(label, fontsize=9, fontweight='bold')
    ax.axis('off')

# ── Row 2: Fuel model + terrain ─────────────────────────────────────────────
ax_fuel = fig.add_subplot(gs[1, 0:2])
if 'fuel_model' in ds.data_vars:
    from src.analysis import _fbfm40_group_colormap
    arr = ds['fuel_model'].values.astype(float)
    arr[arr <= -9000] = np.nan
    rgba, legend = _fbfm40_group_colormap(arr)
    ax_fuel.imshow(rgba, origin='upper', interpolation='nearest')
    patches = [mpatches.Patch(facecolor=c, label=lbl) for lbl, c in legend.items()]
    ax_fuel.legend(handles=patches, loc='lower right', fontsize=6, framealpha=0.85)
ax_fuel.set_title('FBFM40 Fuel Model (surface fuels)', fontsize=10, fontweight='bold')
ax_fuel.axis('off')

for col_idx, (var, label, cmap) in enumerate([
    ('slope',  'Slope (°)',    'Oranges'),
    ('aspect', 'Aspect (° N)', 'hsv'),
], start=2):
    ax = fig.add_subplot(gs[1, col_idx])
    if var in ds.data_vars:
        arr = ds[var].values.astype(float)
        arr[arr <= -9000] = np.nan
        vkw = {'vmin': 0, 'vmax': 360} if var == 'aspect' else {
            'vmin': 0, 'vmax': np.nanpercentile(arr, 98)}
        ax.imshow(arr, cmap=cmap, origin='upper', interpolation='bilinear', **vkw)
    ax.set_title(label, fontsize=9, fontweight='bold')
    ax.axis('off')

# ── Row 3: Fire-weather summary panels ──────────────────────────────────────
for col_idx, (col, label, cmap, unit) in enumerate([
    ('temp_C',   f'Temperature\n{ERA5_DATE}', 'RdYlBu_r', '°C'),
    ('rh_pct',   'Rel. Humidity',              'BrBG',     '%'),
    ('wind_spd', 'Wind Speed',                 'PuBu',     'm/s'),
    ('vpd_hPa',  'VPD',                        'YlOrRd',   'hPa'),
]):
    ax = fig.add_subplot(gs[2, col_idx])
    if col in df.columns and not df[col].isna().all():
        sub = df.dropna(subset=[col]).sample(min(5000, len(df)), random_state=0)
        if 'x' in sub.columns:
            sc = ax.scatter(sub['x'], sub['y'], c=sub[col], s=0.3,
                            cmap=cmap, rasterized=True)
            plt.colorbar(sc, ax=ax, shrink=0.75, label=unit)
        else:
            ax.text(0.5, 0.5, f'{df[col].mean():.1f} {unit}\n(spatially uniform)',
                    ha='center', va='center', transform=ax.transAxes, fontsize=10)
    ax.set_title(label, fontsize=9, fontweight='bold')
    ax.axis('off')

fig.suptitle(
    'Fire-Behavior Environment: Vegetation → Fuels + Terrain + Weather\n'
    f'Northern Colorado Mountain Landscape  |  {ERA5_DATE} {ERA5_HOUR:02d}:00 UTC (ERA5)',
    fontsize=12, fontweight='bold',
)

out = FIGURES_DIR / 'integrated_fire_environment.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

![Integrated fire-behavior environment: vegetation, fuels, terrain, and ERA5 weather](../figures/integrated_fire_environment.png)

## 8  What This Is — and What It Is Not

### What we have built

A description of the fire-behavior environment as of one afternoon in August 2020:

- **LANDFIRE layers (static, landscape-scale):** vegetation type, cover, height, fuel model, disturbance — representing the fuel landscape as of LF2022.
- **Terrain:** elevation, slope, aspect — physical conditioning of fire behavior.
- **ERA5 (dynamic, daily/hourly):** temperature, humidity, wind, VPD — the atmospheric state that drives fuel moisture and fire spread rate.

### What it is NOT

- **Not a fire risk map.** No fuel model is inherently "high risk." Fire risk depends on the combination of fuel load, fuel moisture, wind, slope, and ignition probability — none of which are combined here into a risk score.
- **Not a fire-behavior simulation.** Scott & Burgan fuel models are *inputs* to models like Rothermel's surface fire spread model (implemented in FARSITE, FlamMap, etc.). They are not outputs.
- **Not a prediction.** The 2020 Cameron Peak Fire was driven by specific synoptic and mesoscale conditions not fully captured by ERA5 at 0.25° resolution.

### Limitations

1. **ERA5 resolution (~27.8 km)** is very coarse relative to the 30 m LANDFIRE grid. A production analysis would use dynamically downscaled weather (e.g. WRF, HRRR) or station-interpolated data.
2. **Fuel moisture is not encoded in FBFM40.** The fuel model specifies potential fire behavior under standard moisture scenarios. Actual fuel moisture is a dynamic variable that must come from weather-based models or field measurements.
3. **LF2022 captures landscape state through 2022.** Post-2022 disturbance (logging, fire, insect mortality) is not reflected.
4. **Validation is not performed.** This is an exploratory analysis, not a calibrated or validated model.

## 9  Summary

This two-notebook workflow demonstrates the full conceptual chain:

| Component | Source | Role in fire behavior |
|---|---|---|
| EVT | LANDFIRE LF2022 | Vegetation type — ecological context for fuel assignment |
| EVC + EVH | LANDFIRE LF2022 | Vegetation structure — canopy cover and height |
| FBFM40 | LANDFIRE LF2022 | Surface fuel representation for fire-behavior models |
| Disturbance | LANDFIRE LF2022 | Recent change events that alter fuel structure |
| Elevation | USGS 3DEP | Drives vegetation and fuel type through lapse rate |
| Slope | Derived | Directly accelerates fire spread (Rothermel) |
| Aspect | Derived | Controls drying rate, fuel moisture, and fire exposure |
| Temperature, RH, Wind, VPD | ERA5 | Atmospheric forcing driving fuel moisture and spread rate |

The key conceptual distinction:
- **LANDFIRE** represents the landscape fuel state (relatively static, updated annually).
- **ERA5** represents the atmospheric forcing (dynamic, hourly).
- Neither alone describes fire behavior — both are necessary inputs to a fire simulation model.

---
*This notebook is part of the `landfire-exploration` portfolio project.*